# RI-JK RHF Hessian 分解概览

In [1]:
from pyscf import gto, scf, lib
import numpy as np
from pyscf.hessian import rhf as rhf_hess
from pyscf.df.hessian import rhf as df_rhf_hess
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = scf.RHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_r_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_r_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_r_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


## Hessian 分解概览

在这份问答功能中，我们将只作非常粗略的 Hessian 分解说明。后续的文档会更详细地介绍 Hessian 分解的细节。

对于 Hessian 分解，其大致分为下述部分 (计算量或复杂程度大致从小到大)：

1. 密度矩阵非依赖项。这里是指原子核排斥能的导数。
2. 密度矩阵一阶项导数。自洽场方法下，密度矩阵一阶项一般统括为 hcore。
3. 重叠积分导数贡献。这部分是来自于 U 矩阵的占据部分化简而来的能量加权密度矩阵部分。
4. 复杂能量二阶 Skeleton 导数。在 Hartree-Fock 中，这主要 J/K 的贡献。
5. CP-HF/KS 贡献。

In [5]:
# 1. Density matrix independent term
de_nuc = rhf_hess.hess_nuc(mol)

In [6]:
# 2. Density matrix first-order term (hcore contribution)
# 3. Overlap integral derivative contribution
# 2 and 3 are computed together to `e1`.

# auxbasis_response = 0: only orbital derivatives (basis_2nd)
hessobj_aux0 = mf.Hessian()
hessobj_aux0.auxbasis_response = 0
de_1, ej_aux0, ek_aux0 = df_rhf_hess._partial_hess_ejk(hessobj_aux0)

In [7]:
# auxbasis_response = 1: 1st-order aux response (note the hessian contribution is scaled by 0.5 if auxbasis_response == 1)
hessobj_aux1 = mf.Hessian()
hessobj_aux1.auxbasis_response = 1
_, ej_aux1, ek_aux1 = df_rhf_hess._partial_hess_ejk(hessobj_aux1)

In [8]:
# auxbasis_response = 2: full aux response
hessobj_aux2 = mf.Hessian()
hessobj_aux2.auxbasis_response = 2
_, ej_aux2, ek_aux2 = df_rhf_hess._partial_hess_ejk(hessobj_aux2)

In [9]:
# 4. J/K contribution
# note the auxbasis_response == 1 will scale contribution by 0.5, so some complication arises to count contributions.
de_J20 = ej_aux0.copy()
de_J11 = 2.0 * (ej_aux1 - ej_aux0)
de_J02 = ej_aux2 - 2.0 * ej_aux1 + ej_aux0

de_K20 = 2 * ek_aux0.copy()
de_K11 = 2 * 2.0 * (ek_aux1 - ek_aux0)
de_K02 = 2 * (ek_aux2 - 2.0 * ek_aux1 + ek_aux0)

## CPHF 响应

In [10]:
# 5. CP-HF contribution
de_hess_elec = mf_hess.hess_elec()
de_partial = de_1 + ej_aux2 - ek_aux2
de_cphf = de_hess_elec - de_partial

## 总核验

In [11]:
de_sum = de_1 \
         + de_J20 + de_J11 + de_J02 \
         - 0.5 * (de_K20 + de_K11 + de_K02) \
         + de_cphf + de_nuc

print("de_ref == de_sum:", np.allclose(de_ref, de_sum))
print("max abs difference:", np.max(np.abs(de_ref - de_sum)))

de_ref == de_sum: True
max abs difference: 7.391864897954292e-13


最终，我们将这些分量都放到 `nh3_r_hf_decomp.npz` 文件中，用于后续的核验和分析。

In [12]:
dat = dict(np.load("nh3_r_hf.npz"))
dat.update({
    "de_nuc": de_nuc,
    "de_1": de_1,
    "de_J20": de_J20,
    "de_J11": de_J11,
    "de_J02": de_J02,
    "de_K20": de_K20,
    "de_K11": de_K11,
    "de_K02": de_K02,
    "de_cphf": de_cphf,
})
np.savez("nh3_r_hf_decomp.npz", **dat)